In [2]:
import pandas as pd

In [3]:
df=pd.read_csv('rag_sample_qas_from_kis.csv')
df.head()

,ki_topic,ki_text,sample_question,sample_ground_truth
0,Setting Up a Mobile Device for Company Email,**Setting Up a Mobile Device for Company Email...,"""How do I set up my company email on my mobile...",To set up your company email on your mobile de...
1,Resetting a Forgotten PIN,**Resetting a Forgotten PIN**\n\nIf you have f...,"I forgot my PIN, how can I reset it?","Don't worry, I'm here to help To reset your fo..."
2,Configuring VPN Access for Remote Workers,**Configuring VPN Access for Remote Workers**\...,How do I set up VPN access on my laptop so I c...,To set up VPN access on your laptop and access...
3,Troubleshooting Issues with Microsoft Office,**Troubleshooting Issues with Microsoft Office...,"""My Microsoft Word keeps freezing every time I...",I'd be happy to help you troubleshoot the issu...
4,Setting Up a Conference Call on Cisco Webex,"To set up a conference call on Cisco Webex, fo...",How do I set up a conference call on Cisco Web...,To set up a conference call on Cisco Webex wit...


In [4]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq

from langchain_classic.chains import RetrievalQA
from langchain_text_splitters import RecursiveCharacterTextSplitter
# from langchain.document_loaders import TextLoader

c:\Users\samar\anaconda3\envs\rag_try\Lib\site-packages\langchain_core\_api\deprecation.py:26: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
c:\Users\samar\anaconda3\envs\rag_try\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import getpass
import os
import dotenv


In [ ]:
dotenv.load_dotenv()
os.environ["HUGGINGFACEHUB_API_TOKEN"] = os.getenv("HUGGINGFACEHUB_API_TOKEN")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [10]:
docs = [
    Document(
        page_content=f"Question: {row['sample_question']}\nAnswer: {row['sample_ground_truth']}",
        metadata={"source": row['ki_topic'], "id": i}
    )
    for i, row in df.iterrows()
]

In [25]:
print(docs)

[Document(metadata={'source': 'Setting Up a Mobile Device for Company Email', 'id': 0}, page_content='Question: "How do I set up my company email on my mobile device?"\nAnswer: To set up your company email on your mobile device, please follow these steps:\n\n**First, ensure that you have a supported operating system (iOS, Android, or Windows) and a company email account.**\n\n1. **Check if a Mobile Device Management (MDM) profile is required**: If your company requires MDM for mobile devices, ensure that the profile is installed on your device. If you\'re unsure, contact your IT department for assistance.\n2. **Set up your email account**:\n\t* Go to the Settings app on your mobile device.\n\t* Select "Mail" or "Email" (depending on your device\'s operating system).\n\t* Tap "Add Account" or "Create a new account".\n\t* Select "Exchange" or "Corporate" as the account type.\n\t* Enter your company email address and password.\n\t* If prompted, enter the company\'s email server address (e

In [11]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
splits = splitter.split_documents(docs)
print(f"{len(splits)} chunks created")

54 chunks created


In [12]:
hf_embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)

c:\Users\samar\anaconda3\envs\rag_try\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\samar\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling bac

In [13]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(splits, hf_embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

In [22]:
llm = ChatGroq(model_name="llama-3.3-70b-versatile", temperature=0.2)

In [23]:
from langchain_classic.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff",
    return_source_documents=True
)

In [24]:
def ask(query: str):
    result = qa_chain({"query": query})
    print(result["result"])
    for doc in result["source_documents"]:
        print("----")
        print(doc.metadata)
        print(doc.page_content)

ask("How to reset email password?")

I don't know. The provided context only discusses setting up an email account, but it does not include information on how to reset an email password.
----
{'source': 'Configuring Email on an Android Device', 'id': 9}
Give your account a name and set it as the default account. Tap "Done" to complete the setup.

**Important:** If you encounter any issues during the setup process, try restarting the Email app or your device. Ensure that your email account credentials are correct and that your internet connection is stable. If you are still experiencing issues, contact the IT helpdesk for further assistance.
----
{'source': 'Setting Up a Mobile Device for Company Email', 'id': 0}
* Ensure that the "Use SSL/TLS" or "Use secure connection" option is enabled.
	* Set the authentication method to "Username and Password" or "Domain\Username".
	* If prompted, enter your company's email domain (e.g., company.com).
4. **Verify your email account**:
	* Wait for the email account to synchronize with 